# Calculate 7-Day Moving Average Sales

## Real-Time Scenario

An e-commerce company wants to monitor sales trends.

**The business asks:**

> "For every day, calculate the average sales of the last 7 days including today."

**This helps to:**

* Detect sales spikes
* Forecast demand
* Monitor business growth
* Build dashboards



**Sample Data**

|day_id|sales_date|daily_sales|
|---|---|---|
|1|2026-01-01|10000|
|2|2026-01-02|12000|
|3|2026-01-03|9000|
|4|2026-01-04|15000|
|5|2026-01-05|11000|
|6|2026-01-06|14000|
|7|2026-01-07|13000|
|8|2026-01-08|30000|
|9|2026-01-09|16000|
|10|2026-01-10|8000|
|11|2026-01-11|18000|
|12|2026-01-12|17000|
|13|2026-01-13|19000|
|14|2026-01-14|21000|
|15|2026-01-15|20000|


**Expected Output**

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DateType,
    DoubleType
)
from datetime import datetime

daily_sales_data = [
    (1,  datetime.strptime("2026-01-01", "%Y-%m-%d").date(), 10000.00),
    (2,  datetime.strptime("2026-01-02", "%Y-%m-%d").date(), 12000.00),
    (3,  datetime.strptime("2026-01-03", "%Y-%m-%d").date(),  9000.00),
    (4,  datetime.strptime("2026-01-04", "%Y-%m-%d").date(), 15000.00),
    (5,  datetime.strptime("2026-01-05", "%Y-%m-%d").date(), 11000.00),
    (6,  datetime.strptime("2026-01-06", "%Y-%m-%d").date(), 14000.00),
    (7,  datetime.strptime("2026-01-07", "%Y-%m-%d").date(), 13000.00),
    (8,  datetime.strptime("2026-01-08", "%Y-%m-%d").date(), 30000.00),  # Sales spike
    (9,  datetime.strptime("2026-01-09", "%Y-%m-%d").date(), 16000.00),
    (10, datetime.strptime("2026-01-10", "%Y-%m-%d").date(),  8000.00),  # Low-sales day
    (11, datetime.strptime("2026-01-11", "%Y-%m-%d").date(), 18000.00),
    (12, datetime.strptime("2026-01-12", "%Y-%m-%d").date(), 17000.00),
    (13, datetime.strptime("2026-01-13", "%Y-%m-%d").date(), 19000.00),
    (14, datetime.strptime("2026-01-14", "%Y-%m-%d").date(), 21000.00),
    (15, datetime.strptime("2026-01-15", "%Y-%m-%d").date(), 20000.00)
]

daily_sales_schema = StructType([
    StructField("day_id", IntegerType(), False),
    StructField("sales_date", DateType(), False),
    StructField("daily_sales", DoubleType(), False)
])

daily_sales_df = spark.createDataFrame(
    daily_sales_data,
    daily_sales_schema
)

display(daily_sales_df.orderBy("sales_date"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, avg, round, col

seven_day_window = (
    Window
    .orderBy("sales_date")
    .rowsBetween(-6, 0)
)

In [0]:


moving_average_df = (
    daily_sales_df
    .withColumn(
        "seven_day_total",
        sum("daily_sales").over(seven_day_window)
    )
    .withColumn(
        "seven_day_moving_average",
        round(
            avg("daily_sales").over(seven_day_window),
            2
        )
    )
    .orderBy("sales_date")
)

display(moving_average_df)

### Interview Follow-up
- #### Why use rowsBetween(-6,0)?
- #### What if data is missing for some days?
- #### Difference between Running Total and Moving Average?